# Data overview

Start here. This notebook says what is in the database, how much of it there is, how complete it is, and what it cannot be used to answer. The five question notebooks that follow all assume you have read this one.

Everything is scraped from [basketball-reference.com](https://www.basketball-reference.com) and loaded into a local PostgreSQL database, `nba_analysis`. It has two schemas. `processed` is the cleaned source, one table per scraped page, nothing reshaped. `analyst_ready` is the enriched layer built on top of it, with the joins and the derived rates already done. The notebooks query `analyst_ready` and reach into `processed` only when they need something the enriched layer flattens.

## One convention to fix before anything else

**A season is stored as the year it ended.** The 2023-24 season is `2024`. The 2025-26 season is `2026`. Every table follows that rule and every query below depends on it. When a chart needs the printable form there is a `season_label` column holding `'2023-24'`.

This is the single easiest way to get a silently wrong answer out of this database. A query written against the wrong year returns a full set of plausible rows rather than an error, so it is worth reading twice.

In [1]:
import _setup  # noqa: F401

import pandas as pd

from utils.db_utils import run_query
from utils.custom_plots import (
    distribution_plot,
    grouped_box_plot,
    missing_values_plot,
    time_series_plot,
)
from utils.custom_stats import summary_stats

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

## 1. What is in here

`analyst_ready` is five relations: two wide fact tables and three dimensions that say who, which club, and which year. There are no question-specific tables. Every question in this project is a query written against these five.

The two facts do not cover the same stretch of history, and that difference decides what each notebook is allowed to ask.

In [3]:
coverage = run_query("""
    select 'player_season'         as relation,
           'fact'                  as kind,
           count(*)                as n_rows,
           count(distinct season)  as n_seasons,
           min(season_label)       as first_season,
           max(season_label)       as last_season
    from analyst_ready.player_season
    union all
    select 'team_season', 'fact', count(*), count(distinct season),
           min(season_label), max(season_label)
    from analyst_ready.team_season
    union all
    select 'dim_season', 'dimension', count(*), count(distinct season),
           min(season_label), max(season_label)
    from analyst_ready.dim_season
    union all
    select 'dim_player', 'dimension', count(*), null::bigint,
           null::varchar, null::varchar
    from analyst_ready.dim_player
    union all
    select 'dim_team', 'dimension', count(*), null::bigint,
           null::varchar, null::varchar
    from analyst_ready.dim_team
""")

grain = run_query("""
    select count(*)                                     as rows,
           count(distinct (season, player_id))          as player_seasons,
           count(distinct player_id)                    as distinct_players,
           count(distinct team_id)
               filter (where team_id <> 'tot')          as distinct_clubs,
           count(*) filter (where is_multi_team_season) as traded_mid_season
    from analyst_ready.player_season
""")

display(coverage)
display(grain)

,relation,kind,n_rows,n_seasons,first_season,last_season
0,player_season,fact,4466,8.0,2018-19,2025-26
1,team_season,fact,1693,77.0,1949-50,2025-26
2,dim_season,dimension,80,80.0,1946-47,2025-26
3,dim_player,dimension,1985,NaN,NaN,NaN
4,dim_team,dimension,75,NaN,NaN,NaN


,rows,player_seasons,distinct_players,distinct_clubs,traded_mid_season
0,4466,4466,1278,30,623


Two facts, two very different windows.

`team_season` goes back to 1949-50: 1,693 club seasons across 77 years. `player_season` only covers 2018-19 through 2025-26, because individual box scores were scraped for those eight seasons and no further. So anything about the league as a whole can be asked over three quarters of a century, and anything about individual players cannot go back past 2018-19. Most of the questions in this project sit inside 2020-2026, which is a choice made per question rather than a limit of the data.

The dimensions are wider than either fact. `dim_season` lists 80 seasons starting in 1946-47, and `dim_player` holds 1,985 people. Only 1,278 of those players actually appear in `player_season`. The rest are referenced by an old championship roster or an MVP award and have no box score here.

The grain check matters more than it looks. 4,466 rows and 4,466 distinct season-and-player pairs means **one row per player per season**, with no duplicates to trip over. That is not free. A player traded mid-season gets three rows on the source page, one per club plus a combined line, and 623 of these rows are those combined lines. `analyst_ready` has already picked the right one. Query `processed.player_season_stats` instead if you specifically need his numbers for each club.

In [4]:
rows_by_season = run_query("""
    select 'player_season' as relation, season
    from analyst_ready.player_season
    union all
    select 'team_season', season
    from analyst_ready.team_season
""")

# time_series_plot needs a datetime axis; a season is only an integer year here.
rows_by_season["season_ending"] = pd.to_datetime(
    rows_by_season["season"].astype(str) + "-07-01"
)

# Log scale: one line runs 11-30, the other 530-605.
fig_coverage = time_series_plot(
    rows_by_season,
    date_col="season_ending",
    group_col="relation",
    agg="count",
    freq="YS",
    log_y=True,
    title="Rows per season in each fact table",
)
fig_coverage.show()

The y-axis is logarithmic, because otherwise one line would flatten the other.

The red line is the number of clubs playing each season. Seventeen in 1949-50, down to eight through the late 1950s when the early league contracted, then a long climb to the thirty that have played every season since 2004-05. The blue line is players, and it exists only from 2018-19 onward: between 529 and 605 a season.

The gap between the two lines is the shape of this project. Team history is deep and player history is shallow.

## 2. How complete it is

A quick word on what a blank cell means here, because in this database it usually is not a scraping failure.

Three different things produce a null. Some columns are blank because the statistic did not exist yet: nobody recorded blocks before 1973-74, and there was no three-point line before 1979-80. Some are blank because the fact does not apply, so an undrafted player has no draft pick, a player who never went to college has no college, and a player who received no MVP votes has no ballot position. Only the third kind is a real gap, and in this table there is essentially one column of it.

The slice below is nineteen columns picked to span the width of `player_season`: who he was, how much he played, how well he shot, what the advanced metrics say, and what the bio page adds.

In [5]:
player_slice = run_query("""
    select season, age, position, games_played, minutes_played,
           points_per_game, availability,
           field_goal_pct, three_point_pct, free_throw_pct,
           true_shooting_percentage, usage_percentage, turnover_percentage,
           height_cm, weight_kg, college,
           draft_overall_pick, experience_seasons, mvp_rank
    from analyst_ready.player_season
""")

# Postgres numerics arrive as Decimal objects; several plot helpers skip those.
decimal_cols = [
    "points_per_game", "availability", "field_goal_pct", "three_point_pct",
    "free_throw_pct", "true_shooting_percentage", "usage_percentage",
    "turnover_percentage", "height_cm", "weight_kg",
]
player_slice[decimal_cols] = player_slice[decimal_cols].astype(float)

print(player_slice.shape)
fig_missing = missing_values_plot(
    player_slice,
    mode="bar",
    title="Missing values across a representative slice of player_season",
)
fig_missing.show()

(4466, 19)


Read that bar chart from the bottom up and most of it is good news. Age, position, games played, minutes, points per game, availability, usage, height and weight are complete on all 4,466 rows. Nothing that any of the five questions depends on is missing.

Now the tall bars, in order:

`mvp_rank` is blank on 97.9% of rows, and that is the design. Only 93 player-seasons in eight years received an MVP vote at all, so the other 4,373 correctly have no ballot position. Treating this as 98% missing data would be a misreading. It is a flag, not a measurement.

`draft_overall_pick` (26.2%) and `college` (12.9%) are the same kind of thing. 1,172 of these player-seasons belong to men nobody drafted, and 575 to men who never played US college basketball, which is common for international players. Both facts are informative in themselves.

The shooting percentages are blank when a player never attempted that shot: 237 rows never took a three, 229 never took a free throw. The source writes nothing rather than zero, which is correct, since a player who takes no threes does not have a 0% three-point percentage. He has no three-point percentage. Anything that averages these columns has to decide what to do about that.

`experience_seasons` at 32.2% is the one column that is genuinely thin, and it deserves its own look.

In [6]:
experience_gap = run_query("""
    select season_label,
           count(*)                                        as rows,
           count(experience_seasons)                       as recorded,
           count(*) - count(experience_seasons)            as missing,
           round(100.0 * (count(*) - count(experience_seasons))
                 / count(*), 1)                            as pct_missing
    from analyst_ready.player_season
    group by season, season_label
    order by season
""")
experience_gap

,season_label,rows,recorded,missing,pct_missing
0,2018-19,530,183,347,65.5
1,2019-20,529,247,282,53.3
2,2020-21,540,294,246,45.6
3,2021-22,605,349,256,42.3
4,2022-23,539,394,145,26.9
5,2023-24,572,457,115,20.1
6,2024-25,569,524,45,7.9
7,2025-26,582,581,1,0.2


The gradient is the whole explanation. 65.5% missing in 2018-19, 0.2% in 2025-26, and a clean slope in between.

There is no per-season experience figure on the box-score page, so this column is worked backwards from a career total scraped once, at the time of the scrape: subtract the number of seasons since. That arithmetic needs nothing but the career total for the most recent season, which is why 2025-26 is complete, and it needs more assumptions the further back you go, which is why 2018-19 is two thirds empty. It also counts seasons played rather than calendar years, so a player who missed a full year through injury comes out too low.

The practical consequence for the notebooks that follow: **`experience_seasons` is usable for recent seasons and unreliable for old ones.** `processed.rosters` carries the figure the source states directly, per player per season, for every club from 2018-19 onward. Any question that turns on experience should prefer that column and say which one it used.

## 3. Who the players are

Four attributes carry most of the weight in the questions that follow: height, weight, age, and position. Three of the five ask about height directly or about a ratio built from it. So it is worth knowing what these look like across all 4,466 player-seasons before any question narrows them down.

In [7]:
attributes = run_query("""
    select height_cm, weight_kg, age, position
    from analyst_ready.player_season
""")
attributes[["height_cm", "weight_kg"]] = (
    attributes[["height_cm", "weight_kg"]].astype(float)
)

print(attributes.shape)
attributes.head()

(4466, 4)


,height_cm,weight_kg,age,position
0,198.1,90.7,25,SG
1,200.7,108.9,28,PF
2,182.9,102.1,22,PG
3,210.8,120.2,25,C
4,205.7,115.7,21,C


In [8]:
fig_attributes = distribution_plot(
    attributes,
    title="Height, weight, age and position across all player-seasons",
    n_cols=2,
    show_kde=True,
)
fig_attributes.show()

In [9]:
summary_stats(attributes, cols=["height_cm", "weight_kg", "age"]).round(2)

,column,n,n_missing,pct_missing,n_unique,mean,ci_low,ci_high,trimmed_mean,median,std,mad,iqr,cv,skew,excess_kurtosis,min,q1,q3,max
0,height_cm,4466,0,0.0,23,199.08,198.84,199.33,199.08,198.1,8.33,7.56,12.7,0.04,0.01,-0.21,170.2,193.0,205.7,228.6
1,weight_kg,4466,0,0.0,107,98.09,97.77,98.41,97.66,97.5,10.94,10.08,14.5,0.11,0.41,-0.02,72.1,90.7,105.2,141.1
2,age,4466,0,0.0,25,25.74,25.62,25.87,25.39,25.0,4.18,4.45,5.0,0.16,0.77,0.24,19.0,23.0,28.0,43.0


The typical NBA player of the last eight seasons is 199 cm and 98 kg, and 25 years old. The shortest is 170.2 cm, the tallest 228.6 cm, so the league spans 58 cm of human being.

Three things in that table are worth carrying forward.

**Height has only 23 distinct values across 4,466 rows.** Basketball-Reference publishes height in whole feet and inches, and this database converts it, so every value sits on a 2.54 cm grid. It behaves like a coarse ordinal scale rather than a continuous measurement. That is fine for a mean, and it is worth remembering before running any test that assumes continuity.

**Height is not really one distribution.** Skew is 0.01 and excess kurtosis is negative, which on paper sounds textbook normal. Look at the histogram instead: the mass piles up around 196 cm, dips, then piles up again around 204 cm. That is five position distributions stacked on top of each other. The next figure separates them, and it explains why the two height questions in this project are mostly position questions.

**Age is right-skewed** (skew 0.77, median 25, mean 25.7). Most players are young and a thin tail of veterans runs out to 43. Any comparison of ages between groups should prefer the median, or use a test that does not assume symmetry.

Positions come out close to even, which is expected, since a squad needs one of each. Shooting guard is the largest share at 25.0% and point guard the smallest at 18.1%.

In [10]:
fig_height_position = grouped_box_plot(
    attributes,
    group_col="position",
    value_col="height_cm",
    sort_by="median",
    ascending=True,
    title="Height by position played that season",
)
fig_height_position.show()

There is the shoulder, pulled apart. Mean height climbs from 189.0 cm at point guard to 209.7 cm at centre, a spread of 20.7 cm across the five roles. The boxes for a point guard and a centre barely share any range at all.

That single fact does most of the work later. If one group of players turns out to be taller than another, the first question is not "are elite players taller" but "does that group contain more centres". A height finding in this project that ignores position is unfinished.

The shooting guards look odd, with a very tight box and a long column of flagged outliers. That is the 2.54 cm grid showing itself. When half the sample sits on two or three adjacent values, the interquartile range collapses and the 1.5-IQR rule starts calling ordinary heights outliers. Nothing is wrong with the data. The outlier rule just does not suit a variable this coarse.

## 4. How the league changed

Everything above is confined to the eight seasons of player data. The team table reaches back to 1949-50, and it is worth spending two figures there, because the narrow windows the later notebooks use are a deliberate choice rather than the edge of the dataset.

Two numbers show it clearly. How much a team scores, and how much of its shooting comes from behind the three-point line.

In [11]:
league = run_query("""
    select season, season_label, team_name,
           points_per_game, three_point_attempt_rate
    from analyst_ready.team_season
    where has_been_played
    order by season, team_id
""")
league[["points_per_game", "three_point_attempt_rate"]] = (
    league[["points_per_game", "three_point_attempt_rate"]].astype(float)
)
league["season_ending"] = pd.to_datetime(league["season"].astype(str) + "-07-01")

print(league.shape)

fig_scoring = time_series_plot(
    league,
    date_col="season_ending",
    value_col="points_per_game",
    agg="mean",
    freq="YS",
    show_points=True,
    title="League average points scored per game, by season",
)
fig_scoring.show()

(1692, 6)


In [12]:
fig_threes = time_series_plot(
    league,
    date_col="season_ending",
    value_col="three_point_attempt_rate",
    agg="mean",
    freq="YS",
    show_points=True,
    title="Share of a team's shots taken from three-point range, by season",
)
fig_threes.show()

Both charts are the same argument from two angles: this database contains several different games, all of them called basketball.

Scoring bottoms out at 79.5 points a game in 1953-54, jumps to 93.1 the next season when the shot clock arrived, and peaks at 118.8 in 1961-62. Then a forty-year slide down to 91.6 in 1998-99, and a climb back to 115.6 in 2025-26. A team scoring 100 points was excellent in 1999 and is below average now.

The three-point rate starts at 3.1% in 1979-80, the first season the line existed. The spike between 1994-95 and 1996-97 is the league's experiment with a shorter arc, which was reversed and shows up as an immediate drop to 16.0% in 1997-98. After that the rise is uninterrupted: 41.5% of all shots in 2025-26 now come from three, up from 22.4% in 2008-09.

Two consequences for the notebooks that follow. Any raw count compared across eras is measuring the era as much as the player, which is why the team table carries rate columns. And the later questions all sit inside 2020-2026 for a reason. Those seasons are close enough together to compare directly without arguing about which game is being played.

## 5. What this data cannot tell us

Four limits. The first constrains the most.

### There is no wins column anywhere in this database

Not in `team_season`, not in `processed.team_season_stats`, not anywhere. Basketball-Reference publishes standings; this scrape did not collect them.

Two columns look like they might be one. `points_rank` is the display order of the source page, which sorts by total points scored. A club ranked first scored the most points that season, and may or may not have won the most games. `win_shares` is worse, because the name is so close: it is an estimate of how many wins a *player's* production was worth, derived from his box score, and it has no connection to any team's actual record.

So nothing in this project can support a sentence about winning. "The clubs that shot best won more" is a claim this data cannot evaluate. "The clubs that shot best scored more" is one it can. That distinction has to survive into the write-up, or the conclusion is simply wrong.

The championship is the one exception. `dim_season.champion_team_name` and `player_season.is_on_champion_team` record who won the title, one club a season, which is a different fact from a win total.

### The MVP ballot only goes back to 2018-19

`processed.mvp_winners` holds one winner a season back to 1955-56. The full ballot, meaning everyone who received a vote, was scraped only for 2018-19 onward: 93 player-seasons across eight years. Every question in this project that uses the ballot sits inside that window, so nothing here is affected. But "has the MVP electorate changed since the 1980s" is not answerable, and neither is anything about players who received votes and never won.

### Player history stops at 2018-19

Eight seasons of individual box scores. `dim_player` lists 1,985 people and only 1,278 of them have a row in `player_season`. Michael Jordan is in this database with a full bio, 198.1 cm, drafted 1984, and zero seasons of statistics, because he retired fifteen years before the scrape window opens. Comparing a modern player to a historic one means comparing a box score to a bio page.

### Experience before a season is partly reconstructed

Covered in section 2. Usable for recent seasons, thin and drifting for older ones, with a better version sitting in `processed.rosters`. Any result that leans on it needs the caveat attached.

---

One smaller thing worth knowing about. `availability` is a share of games, not minutes, so a player who came on for two minutes counts as fully available that night. Two rows in eight seasons also sit slightly above 1.0, which happens when a traded player's two clubs were at different points in their schedules. Both are real, neither is an error.